In [1]:
# На этом этапе мы подготавливаем библиотеку для символьного решения задачи.
# SymPy позволяет хранить формулы точно, брать производные, решать системы
# и подставлять найденные точки без преждевременного округления.
# Восклицательный знак запускает команду оболочки из Jupyter;
# pip устанавливает sympy или сообщает, что библиотека уже установлена.
!pip install sympy


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Подключаем библиотеку SymPy для символьных вычислений,
# автоматического нахождения производных и решения систем уравнений.
import sympy as sp

# Задаём допустимую погрешность при проверке ограничений.
t = 1e-8

# Создаём действительные символьные переменные x и y.
x, y = sp.symbols("x y", real=True)

# Вводим множитель Лагранжа.
lam = sp.symbols("lambda", real=True)

f = 2 * x - y

constraints = [
    x**2 + y**2 - 5
]

variables = [x, y]
multipliers = [lam]


# ---------------------------------------------------------
# 1. ПРОВЕРКА РЕГУЛЯРНОСТИ ОГРАНИЧЕНИЯ
# ---------------------------------------------------------

# Строим матрицу Якоби ограничения.
# Она состоит из частных производных ограничения по x и y.
# Метод jacobian() автоматически вычисляет эти производные.
# В нашей задаче получаем матрицу [2x, 2y].
J = sp.Matrix(constraints).jacobian(variables)

print("Матрица Якоби ограничений:")
sp.pprint(J)


# Проверяем, существуют ли особые допустимые точки.
# Особая точка для одного ограничения — это точка,
# в которой обе частные производные равны нулю.
#
# При этом точка обязательно должна удовлетворять
# исходному ограничению x² + y² = 5.
#
# Поэтому одновременно решаем три уравнения:
# x² + y² - 5 = 0
# 2x = 0
# 2y = 0
#
# constraints содержит исходное ограничение,
# list(J) содержит обе частные производные.
# solve() решает полученную систему.
# dict=True возвращает решения в виде словарей.
singular_solutions = sp.solve(
    constraints + list(J),
    variables,
    dict=True
)


# Если система не имеет решений, значит особых допустимых точек нет.
# Следовательно, во всех допустимых точках градиент ограничения
# не равен нулю, поэтому все эти точки регулярные.
if not singular_solutions:
    print("Особых допустимых точек нет.")
    print("Все допустимые точки регулярные.")
else:
    print("Найдены особые допустимые точки:")
    print(singular_solutions)


# ---------------------------------------------------------
# 2. ПОСТРОЕНИЕ ФУНКЦИИ ЛАГРАНЖА
# ---------------------------------------------------------

# Строим функцию Лагранжа по формуле L = f + λh.
# zip() сопоставляет каждому ограничению свой множитель Лагранжа,
# а sum() складывает все полученные произведения.
#
# В нашей задаче получается:
# L = 2x - y + λ(x² + y² - 5).
L = f + sum(
    a * h
    for a, h in zip(multipliers, constraints)
)


# ---------------------------------------------------------
# 3. СОСТАВЛЕНИЕ СИСТЕМЫ СТАЦИОНАРНОСТИ
# ---------------------------------------------------------

# Вычисляем частные производные функции Лагранжа по x и y.
# sp.diff(L, v) вычисляет производную по переменной v.
#
# Затем добавляем исходное ограничение.
# Все полученные выражения должны быть равны нулю.
#
# Таким образом, формируем систему:
# 2 + 2λx = 0
# -1 + 2λy = 0
# x² + y² - 5 = 0
equations = [
    sp.diff(L, v)
    for v in variables
] + constraints


# Объединяем исходные переменные и множитель Лагранжа.
# В результате получаем список неизвестных [x, y, λ].
unknowns = variables + multipliers


# ---------------------------------------------------------
# 4. РЕШЕНИЕ СИСТЕМЫ ЛАГРАНЖА
# ---------------------------------------------------------

# Решаем систему стационарности с помощью SymPy.
# solve() находит значения x, y и λ, удовлетворяющие системе.
# dict=True возвращает каждое решение в виде словаря.
#
# В результате получаем две точки-кандидата на экстремум.
solutions = sp.solve(
    equations,
    unknowns,
    dict=True
)


# ---------------------------------------------------------
# 5. ВЫВОД ИСХОДНЫХ ДАННЫХ И ПОЛУЧЕННОЙ СИСТЕМЫ
# ---------------------------------------------------------

print("Целевая функция:")
sp.pprint(f)

print("\nОграничения:")
for h in constraints:
    sp.pprint(h)

print("\nФункция Лагранжа:")
sp.pprint(L)

print("\nУравнения стационарности:")
for eq in equations:
    sp.pprint(eq)

print("\nНайденные кандидаты:")


# Создаём пустой список, в который будем добавлять
# только те решения, которые удовлетворяют ограничениям.
valid_solutions = []


# ---------------------------------------------------------
# 6. ПРОВЕРКА НАЙДЕННЫХ КАНДИДАТОВ
# ---------------------------------------------------------

# Перебираем все найденные решения.
# enumerate() позволяет одновременно получить
# номер кандидата и само решение.
for number, solution in enumerate(solutions, start=1):

    # Проверяем, что все найденные значения действительные.
    # sp.Q.real() задаёт условие действительности,
    # а sp.ask() проверяет, выполняется ли оно.
    #
    # all() проверяет все неизвестные: x, y и λ.
    # Если хотя бы для одного значения действительность
    # не подтверждена, пропускаем это решение.
    if not all(
        sp.ask(sp.Q.real(solution[v])) is True
        for v in unknowns
    ):
        continue

    # Создаём словарь с координатами найденной точки.
    # Например, для точки (2, -1) получим {x: 2, y: -1}.
    # Этот словарь используем для подстановки значений в формулы.
    point_subs = {
        v: solution[v]
        for v in variables
    }


    # -----------------------------------------------------
    # 7. ВЫЧИСЛЕНИЕ НЕВЯЗОК
    # -----------------------------------------------------

    # Невязка показывает, насколько найденная точка
    # отклоняется от выполнения исходного ограничения.
    #
    # Подставляем координаты точки в каждое ограничение.
    # subs() выполняет подстановку,
    # simplify() упрощает полученное выражение.
    #
    # Если невязка равна нулю, ограничение выполняется точно.
    residuals = [
        sp.simplify(h.subs(point_subs))
        for h in constraints
    ]


    # -----------------------------------------------------
    # 8. ПРОВЕРКА ДОПУСТИМОСТИ
    # -----------------------------------------------------

    # Проверяем, выполняются ли ограничения с заданной точностью.
    #
    # float() преобразует невязку в число.
    # abs() вычисляет её модуль, поскольку отклонение
    # может быть как положительным, так и отрицательным.
    #
    # Сравниваем модуль каждой невязки с погрешностью t.
    # all() проверяет, что все ограничения выполнены.
    is_feasible = all(
        abs(float(r)) <= t
        for r in residuals
    )


    # -----------------------------------------------------
    # 9. ВЫЧИСЛЕНИЕ ЗНАЧЕНИЯ ЦЕЛЕВОЙ ФУНКЦИИ
    # -----------------------------------------------------

    # Подставляем координаты кандидата в целевую функцию.
    # Полученное значение понадобится для сравнения
    # найденных кандидатов и определения экстремумов.
    value = sp.simplify(f.subs(point_subs))


    # -----------------------------------------------------
    # 10. ВЫВОД РЕЗУЛЬТАТОВ
    # -----------------------------------------------------

    print(f"\n{'─' * 35}")
    print(f"Кандидат {number}")
    print(f"{'─' * 35}")

    # Выводим координаты точки.
    # tuple() собирает значения x и y в кортеж вида (x, y).
    print("Точка:", tuple(solution[v] for v in variables))

    print("Множитель Лагранжа:", solution[lam])
    print("Невязки:", residuals)

    # Если точка допустима, выводим "да", иначе — "нет".
    print("Допустимость:", "да" if is_feasible else "нет")

    print("Значение функции:", value)


    # Сохраняем только допустимые решения.
    # В дальнейшем по ним будем определять
    # минимальное и максимальное значения функции.
    if is_feasible:
        valid_solutions.append(solution)

Матрица Якоби ограничений:
[2⋅x  2⋅y]
Особых допустимых точек нет.
Все допустимые точки регулярные.
Целевая функция:
2⋅x - y

Ограничения:
 2    2    
x  + y  - 5

Функция Лагранжа:
  ⎛ 2    2    ⎞          
λ⋅⎝x  + y  - 5⎠ + 2⋅x - y

Уравнения стационарности:
2⋅λ⋅x + 2
2⋅λ⋅y - 1
 2    2    
x  + y  - 5

Найденные кандидаты:

───────────────────────────────────
Кандидат 1
───────────────────────────────────
Точка: (2, -1)
Множитель Лагранжа: -1/2
Невязки: [0]
Допустимость: да
Значение функции: 5

───────────────────────────────────
Кандидат 2
───────────────────────────────────
Точка: (-2, 1)
Множитель Лагранжа: 1/2
Невязки: [0]
Допустимость: да
Значение функции: -5


In [2]:
# На этом этапе сравниваем значения цели среди отобранных кандидатов.
# Создаём список пар «точка — значение функции» для итогового выбора.
# Метод Лагранжа сам по себе не гарантирует глобальность экстремумов.
# Здесь она обоснована отдельно в следующей Markdown-ячейке:
# окружность компактна, f непрерывна, все точки регулярны,
# а полный список решений системы содержит все возможные экстремумы.
values = []

# Перебираем только действительные решения, уже прошедшие проверку допустимости.
for solution in valid_solutions:
    # Создаём словарь координат для подстановки в исходную целевую функцию.
    point = {
        # Связываем символ x с первой координатой решения.
        x: solution[x],
        # Связываем символ y со второй координатой решения.
        y: solution[y]
    }

    # subs подставляет точку в f = 2x - y; simplify упрощает точное значение.
    value = sp.simplify(f.subs(point))

    # Добавляем запись в список, сохраняя вместе координаты и значение цели.
    values.append({
        # Ключ point хранит кортеж (x, y), чтобы затем назвать точку экстремума.
        "point": (solution[x], solution[y]),
        # Ключ value хранит вычисленное значение f для сравнения кандидатов.
        "value": value
    })

# Выбираем запись с наименьшим значением f среди кандидатов.
# key задаёт критерий: lambda item возвращает поле value текущей записи.
# Это анонимная функция Python, не множитель Лагранжа lam.
min_candidate = min(values, key=lambda item: item["value"])
# Аналогично выбираем запись с наибольшим значением f по тому же полю value.
max_candidate = max(values, key=lambda item: item["value"])

# Выводим итог сравнения; слова «глобальный» опираются на отдельное доказательство.
print("\nИтог:")
# Начинаем сообщение о глобальном минимуме.
print("Глобальный минимум:",
      # Из выбранной записи берём минимальное значение функции: -5.
      min_candidate["value"],
      # Добавляем текст, связывающий значение функции с координатами точки.
      "в точке",
      # Выводим точку достижения минимума (-2, 1) и завершаем вызов print.
      min_candidate["point"])

# Начинаем сообщение о глобальном максимуме.
print("Глобальный максимум:",
      # Из выбранной записи берём максимальное значение функции: 5.
      max_candidate["value"],
      # Добавляем пояснение, что далее следуют координаты точки.
      "в точке",
      # Выводим точку достижения максимума (2, -1) и завершаем вызов print.
      max_candidate["point"])


Итог:
Глобальный минимум: -5 в точке (-2, 1)
Глобальный максимум: 5 в точке (2, -1)


### Обоснование глобальности экстремумов

Рассматривается задача оптимизации функции

$$
f(x,y)=2x-y
$$

при ограничении

$$
x^2+y^2=5.
$$

**1. Компактность допустимого множества**

Допустимое множество

$$
D=\{(x,y)\in\mathbb{R}^2 \mid x^2+y^2=5\}
$$

представляет собой окружность радиуса $\sqrt{5}$.

Окружность является замкнутым и ограниченным множеством,
следовательно, она компактна.

**2. Непрерывность целевой функции**

Функция $f(x,y)=2x-y$ является линейной и непрерывной
на всей плоскости $\mathbb{R}^2$.

По теореме Вейерштрасса непрерывная функция на компактном
множестве достигает глобального минимума и максимума.

**3. Полнота найденных кандидатов**

Все допустимые точки регулярны, поскольку

$$
\nabla h(x,y)=(2x,2y)\neq(0,0)
$$

на окружности $x^2+y^2=5$.

При решении системы Лагранжа найдены все действительные
точки-кандидаты:

$$
P_1=(-2,1), \qquad P_2=(2,-1).
$$

**4. Сравнение значений функции**

$$
f(-2,1)=2(-2)-1=-5,
$$

$$
f(2,-1)=2(2)-(-1)=5.
$$

Следовательно, глобальные экстремумы:

- **Глобальный минимум:** $f_{\min}=-5$ в точке $(-2,1)$.
- **Глобальный максимум:** $f_{\max}=5$ в точке $(2,-1)$.

Таким образом, глобальность найденных экстремумов
обоснована компактностью допустимого множества,
непрерывностью целевой функции и полнотой списка кандидатов.